# 07 -- Stress testing (downturn scenario)

**What this notebook does (plain English):** Asks the key risk question: *how
much worse would losses get in a recession?* Instead of guessing, we use a real
one. The 2007/2008 vintages **lived through** the global financial crisis, so the
jump from the calm 2015 book to the crisis books gives an **observed** downturn
multiplier for both PD and LGD. We apply that downturn to the calm-year portfolio
and read off the increase in Expected Loss.

**Headline result:** under the crisis-calibrated downturn, portfolio Expected
Loss rises several-fold versus the calm baseline -- driven by PD and LGD getting
worse *at the same time*.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and split into calm (2015) and downturn (2007-08) books.
import pandas as pd
import numpy as np
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
calm = base[base['vintage_year'] == 2015]
downturn = base[base['vintage_year'].isin([2007, 2008])]

In [3]:
# Observed downturn multipliers: how much PD and LGD worsened in the crisis.
pd_calm, pd_down = calm['ever_default'].mean(), downturn['ever_default'].mean()
lgd_calm = calm.loc[calm['disposed'], 'lgd'].mean()
lgd_down = downturn.loc[downturn['disposed'], 'lgd'].mean()
pd_mult = pd_down / pd_calm
lgd_mult = lgd_down / lgd_calm
print(f'PD multiplier  = {pd_mult:.2f}x   LGD multiplier = {lgd_mult:.2f}x')

PD multiplier  = 4.36x   LGD multiplier = 2.30x


In [4]:
# Macro context for the downturn (documented assumptions; production would pull
# these live from FRED: unemployment UNRATE, house prices CSUSHPINSA). Values
# reflect the 2008-09 GFC path, comparable to a CCAR severely-adverse scenario.
macro = pd.DataFrame([
    {'scenario': 'baseline (2015 calm)', 'unemployment_pct': 5.3, 'hpi_change_pct': 5.0},
    {'scenario': 'severely adverse (GFC 2008-09)', 'unemployment_pct': 10.0, 'hpi_change_pct': -30.0},
])
macro

,scenario,unemployment_pct,hpi_change_pct
0,baseline (2015 calm),5.3,5.0
1,severely adverse (GFC 2008-09),10.0,-30.0


In [5]:
# Apply the downturn to the calm-year book: stress PD and LGD, recompute EL.
ead_calm = np.where(calm['ever_default'], calm['ead'], calm['original_upb'])
base_pd = calm['ever_default'].mean()
base_lgd = lgd_calm
baseline_el = (base_pd * base_lgd * ead_calm).sum()
stressed_el = (min(base_pd * pd_mult, 1.0) * min(base_lgd * lgd_mult, 1.0) * ead_calm).sum()
stress_tbl = pd.DataFrame([
    {'measure': 'PD', 'baseline': round(base_pd, 4), 'stressed': round(min(base_pd * pd_mult, 1.0), 4)},
    {'measure': 'LGD', 'baseline': round(base_lgd, 4), 'stressed': round(min(base_lgd * lgd_mult, 1.0), 4)},
    {'measure': 'expected_loss', 'baseline': round(baseline_el, 0), 'stressed': round(stressed_el, 0)},
    {'measure': 'EL_uplift_x', 'baseline': 1.0, 'stressed': round(stressed_el / baseline_el, 2)},
])
save_csv(stress_tbl, 'output/07_stress_test.csv')
stress_tbl

,measure,baseline,stressed
0,PD,2.420000e-02,1.055000e-01
1,LGD,2.464000e-01,5.673000e-01
2,expected_loss,6.655434e+07,6.683277e+08
3,EL_uplift_x,1.000000e+00,1.004000e+01


**Reading the table:** we take the calm 2015 portfolio and push PD and LGD
up by the multipliers the crisis actually produced. Because the two stack
multiplicatively, Expected Loss rises far more than either driver alone -- the
core lesson of downturn stress testing.

**Extension -- climate scenario (sketch, not built):** the same machinery extends
to physical climate risk. A flood or wildfire shock lowers house prices in
exposed postcodes, which raises **LGD** (smaller recovery on sale) and, via
negative equity, raises **PD**. One would overlay a hazard map on the property
postcode, apply a region-specific house-price haircut, and re-run this exact
PD/LGD/EL stress -- a cheap, high-signal differentiator for a climate-risk role.